In the previous section we did vector search by hand with numpy. We embedded the query, computed dot products, and found the best matches. Writing the argsort and matrix code every time gets old, and it can't filter by course. So instead we'll use a library that wraps all of it.

We'll use minsearch, the small in-memory search library we already used in module 1 for text search. It has a VectorSearch class for vector search.

Both classes share the same API:

- `fit` to index data
- `search` to query
- `filter_dict` in search to filter by keyword

It's the simplest way to get started with vector search.

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

len(texts)


1368

In [5]:
# First we import tqdm to see the progress
from tqdm import tqdm

# Next we chunk the texts into batches of 50 and encode each batch

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

# We end up with 1368 vectors.

  0%|          | 0/28 [00:00<?, ?it/s]

100%|██████████| 28/28 [00:09<00:00,  2.86it/s]


1368

In [7]:
import numpy as np
X = np.array(vectors)

X.shape

(1368, 384)

### Creating the index
---

We already have our documents and vectors from the previous section.

Index them:


In [9]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

We pass the numpy array X with all embeddings and the list of documents as payload. The keyword_fields parameter works the same as in the text Index, so we can filter by course later.

### Searching
---

Let's search for a question

In [10]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

Under the hood it does the same thing we just did by hand. It computes the dot product between each vector (after filtering) and our query vector.

Look at the top result:

In [12]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

### Filtering by course
---

Like the text index, we can filter by keyword fields. This matters for user experience. A student in LLM Zoom Camp doesn't care about answers from the data engineering course. So we narrow to their course first, then score only within it.

Pass a `filter_dict`:

In [14]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co